# 08 · Revision run — every remaining number, table and figure in one pass

One notebook, one detector load, one staging pass. Produces everything the revisions need,
then trains Baseline 3 unattended at the end.

## Nothing here is blocked on anyone

As of 2026-09-12 the training recipe, the train/val/test split and the original predictor
harness are all recovered. Every open checklist item is now ours to close.

| what was blocked | where it came from |
|---|---|
| training recipe | `train_baseline.py` — COCO init, batch 2, seed 42, SGD 0.005/0.9/5e-4, StepLR 3/0.1 |
| train/val/test split | `splits.csv` / `baseline1_splits.csv`, 77,012 B — 907/113/114 annotated, 3,748 clean |
| preprocessing | `algoverse_dataset.py` — max-normalise at native resolution, native-pixel boxes |
| FROC definition | `froc_eval.py` — greedy one-to-one, clean images in the denominator |
| predictor label + protocol | `config.json` — 419 images, 23 positives, `edit_norm > 0.4919`, liblinear, KFold over chests |

One thing still needs a person, and it is a question rather than a blocker: **where the 0.889
sensitivity in the earlier draft came from.** No recovered script produces it and both
definitional differences push sensitivity down, not up. Most likely the published NODE21
challenge baseline was quoted as though it described this checkpoint. Until someone confirms,
it does not go in the paper.

## Order, and why

Cheap checklist closures first so numbers land early, the 50-minute clean-image run in the
middle, figures once the data exists, Baseline 3 last so it can run unattended.

| § | what | cost | closes |
|---|---|---|---|
| 1–2 | paths, detector, split, environment capture | 2 min | reproducibility |
| 3 | copy-paste twins, paired generator test | 10 min | **item 3** |
| 4 | k-NN coverage | 1 min | **item 9** |
| 5 | all 3,748 clean images scored | ~50 min | true FP rate |
| 6 | Baseline 1 FROC, `froc_eval` convention, 4 scopes | 3 min | **item 10** |
| 7 | Baseline 2 abstention, image-level + clustered | 1 min | **items 7, 10** |
| 8 | real-only predictor power check | 1 min | **item 8** |
| 9 | every figure and table | 3 min | — |
| 10 | manifest, lockfile, checklist summary | 1 min | reproducibility |
| 11 | Baseline 3 training, unattended | 6–10 h | — |

Everything resumable: detections sync to Drive every 50 images, checkpoints every epoch.
Sections 3–10 finish inside one session. Re-running a completed section is a no-op.

## 1 · Paths, split, detector — loaded once

In [ ]:
# ===========================================================================
# CANONICAL DRIVE PATHS
# Mirrored from src/paths.py and notebooks/CONFIG_CELL.md. Mapped from Drive
# 2026-09-12 with the Drive connector. Change all three together.
# Full layout, folder ids and the old->new table: DRIVE_LAYOUT.md
# ===========================================================================
!pip -q install SimpleITK
import os, json, glob, time, shutil, subprocess, random
from pathlib import Path
from google.colab import drive

# --- mount -----------------------------------------------------------------
# ismount(), not isdir(). A plain local directory under an unmounted
# /content/drive is also a dir, and creating one blocks the mount and makes
# Drive look empty -- that happened once and looked like a wiped Drive.
if not os.path.ismount('/content/drive'):
    if Path('/content/drive').exists():
        os.system('fusermount -u /content/drive 2>/dev/null')
        shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'mount failed'

# --- resolve the root ------------------------------------------------------
# MyDrive/Algoverse is a SHORTCUT to the shared folder Feliciano_Algoverse.
# One folder, not two; the FUSE mount resolves it as a directory.
# Test for the 01_data marker, never for the root itself: an unresolved
# shortcut and a stale empty directory both "exist", and that is exactly how
# a stray empty results/ tree got created on 2026-09-12.
ROOT = None
for _c in ['/content/drive/MyDrive/Algoverse',
           '/content/drive/MyDrive/Feliciano_Algoverse',
           '/content/drive/Shareddrives/Feliciano_Algoverse']:
    if (Path(_c)/'01_data').is_dir():
        ROOT = Path(_c); break
assert ROOT is not None, (
    'Algoverse root not found. Tried MyDrive/Algoverse, '
    'MyDrive/Feliciano_Algoverse, Shareddrives/Feliciano_Algoverse.\n'
    f'MyDrive top level: '
    f'{sorted(p.name for p in Path("/content/drive/MyDrive").iterdir())[:20]}\n'
    'MyDrive/Algoverse is a shortcut to the shared Feliciano_Algoverse. If it '
    'is gone: Drive -> Shared with me -> right-click Feliciano_Algoverse -> '
    'Add shortcut to Drive -> My Drive.')

# --- layout ----------------------------------------------------------------
SOURCE    = ROOT/'01_data'/'00_source'           # node21, chexpert, mimic_cxr
NODE21    = SOURCE/'node21'
MHA_SRC   = NODE21/'images'                      # 4,882 .mha
ANN_CSV   = NODE21/'metadata.csv'                # 5,224 rows, 1,476 label==1
GRID      = ROOT/'01_data'/'01_grid'
GRID_CSV  = GRID/'grid_v5.csv'                   # 231,145 bytes if it is the right one
RUNS      = GRID/'_runs'                         # generation checkpoint zips, not data
EMB_DIR   = ROOT/'01_data'/'02_embeddings'
CPASTE    = ROOT/'01_data'/'03_copypaste'
MODELS    = ROOT/'02_results'/'00_models'
CKPT      = MODELS/'baseline1_checkpoint.pth'    # .pth -- the old .pt path is dead
FIGS      = ROOT/'02_results'/'01_figures'
B1_DIR    = ROOT/'02_results'/'02_baselines'
PRED_DIR  = ROOT/'02_results'/'03_predictor'
B3_DIR    = ROOT/'02_results'/'04_baseline3'

print(f'root: {ROOT}')

import numpy as np, pandas as pd, SimpleITK as sitk, torch, torchvision, cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

REV = ROOT/'02_results'/'06_revision'
for d in [REV, REV/'tables', REV/'figures', REV/'dets_native', REV/'dets_copypaste']:
    d.mkdir(parents=True, exist_ok=True)
LOC = Path('/content/rev')
for d in [LOC, LOC/'dets_native', LOC/'dets_copypaste', LOC/'node21']:
    d.mkdir(parents=True, exist_ok=True)

for _p in (CKPT, ANN_CSV, MHA_SRC, GRID_CSV, CPASTE/'copypaste.csv', EMB_DIR):
    assert _p.exists(), f'{_p} missing -- see DRIVE_LAYOUT.md'

SPLIT_CSV = None
for _c in [SOURCE/'baseline1_splits.csv', SOURCE/'splits.csv',
           NODE21/'baseline1_splits.csv', NODE21/'splits.csv']:
    if _c.exists(): SPLIT_CSV = _c; break
if SPLIT_CSV is None:
    _h = sorted(glob.glob('/content/drive/MyDrive/**/baseline1_splits.csv', recursive=True)
                + glob.glob('/content/drive/MyDrive/**/splits.csv', recursive=True)
                + glob.glob('/content/*splits.csv'))
    assert _h, (f'splits.csv not found. Put it at {SOURCE}/baseline1_splits.csv -- '
                '77,012 bytes, in Maia\'s Drive folder and in Dhruv\'s predictor bundle.')
    SPLIT_CSV = Path(_h[0])
SPL = pd.read_csv(SPLIT_CSV)
SPLIT_OF = dict(zip(SPL.img_name, SPL.split))
print(f'split: {SPLIT_CSV.name} ({SPLIT_CSV.stat().st_size:,} B)')
print(SPL.split.value_counts().to_string())

raw = pd.read_csv(ANN_CSV)
raw = raw[raw.img_name != 'n0507.mha']       # byte-identical duplicate of n1059
GTB = {n: [[float(r.x), float(r.y), float(r.x)+float(r.width), float(r.y)+float(r.height)]
           for r in g.itertuples()]
       for n, g in raw[raw.label == 1].groupby('img_name')}
ANNOT = sorted(GTB)
CLEAN = sorted(set(SPL.img_name) - set(raw.img_name) - {'n0507.mha'})
print(f'\n{len(ANNOT)} annotated ({sum(len(v) for v in GTB.values())} nodules), '
      f'{len(CLEAN)} clean')

SCORE_MIN, DET_SIZE, SIZE = 0.05, 800, 512
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
DET = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=None, weights_backbone=None, box_score_thresh=0.0, box_detections_per_img=300)
DET.roi_heads.box_predictor = FastRCNNPredictor(
    DET.roi_heads.box_predictor.cls_score.in_features, 2)
_st = torch.load(CKPT, map_location=DEV, weights_only=False)
DET.load_state_dict(_st['model'] if isinstance(_st, dict) and 'model' in _st else _st)
DET = DET.eval().to(DEV)
print(f'\ndetector on {DEV} from {CKPT.name}; writing to {REV}')

## 2 · Environment capture

`requirements.txt` in the repo has no pins, which is one of the five reproducibility
blockers. This writes a real lockfile from the session that produces the numbers, alongside
the checkpoint md5 and the input hashes.

In [ ]:
import hashlib, subprocess, sys, platform, time, shutil, json


def md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


lock = subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                      capture_output=True, text=True).stdout
(REV/'requirements-lock.txt').write_text(lock)

ENV = dict(
    date=time.strftime('%Y-%m-%d %H:%M:%S'),
    python=sys.version.split()[0], platform=platform.platform(),
    torch=torch.__version__, torchvision=torchvision.__version__,
    cuda=torch.version.cuda, gpu=(torch.cuda.get_device_name(0)
                                  if torch.cuda.is_available() else None),
    numpy=np.__version__, pandas=pd.__version__,
    checkpoint=str(CKPT), checkpoint_md5=md5(CKPT),
    split=str(SPLIT_CSV), split_md5=md5(SPLIT_CSV),
    grid=str(GRID_CSV), grid_md5=md5(GRID_CSV),
    n_annotated=len(ANNOT), n_clean=len(CLEAN),
    n_nodules=int(sum(len(v) for v in GTB.values())),
    score_min=SCORE_MIN, notes='n0507 excluded: byte-identical duplicate of n1059')
json.dump(ENV, open(REV/'environment.json', 'w'), indent=2)
for k in ['date', 'gpu', 'torch', 'checkpoint_md5', 'split_md5', 'grid_md5']:
    print(f'  {k:16} {ENV[k]}')
print(f'\nrequirements-lock.txt: {len(lock.splitlines())} pinned packages')
print(f'checkpoint md5 {ENV["checkpoint_md5"]}'
      + ('  <-- matches MANIFEST.md' if ENV['checkpoint_md5'] ==
         '710179c54d6658f463ff453522de2a54' else '  <-- DOES NOT match MANIFEST.md'))

## 2b · Shared machinery

Defined once and reused by every section: native preprocessing, native inference,
`froc_eval.py`'s greedy matcher, and Drive sync.

In [ ]:
def load_native(path):
    """algoverse_dataset.NoduleDataset preprocessing, exactly: squeeze, max-normalise."""
    a = np.asarray(sitk.GetArrayFromImage(sitk.ReadImage(str(path)))).squeeze()
    if a.ndim != 2:
        raise ValueError(f'expected 2D after squeeze, got {a.shape}')
    m = float(a.max()) if a.max() > 0 else 1.0
    return a.astype(np.float32)/m


@torch.no_grad()
def detect_native(a):
    """No resize -- GeneralizedRCNNTransform scales internally, as in training."""
    t = torch.from_numpy(a)[None].repeat(3, 1, 1)
    o = DET([t.to(DEV)])[0]
    return o['boxes'].cpu().numpy(), o['scores'].cpu().numpy()


@torch.no_grad()
def detect_512(a01, size=DET_SIZE):
    """Grid-parity path: 512 canvas up to 800, fractional boxes."""
    from torchvision.transforms import functional as TF
    im = Image.fromarray((a01*255).astype(np.uint8)).convert('RGB') \
              .resize((size, size), Image.LANCZOS)
    o = DET([TF.to_tensor(im).to(DEV)])[0]
    return o['boxes'].cpu().numpy()/size, o['scores'].cpu().numpy()


def center_in_box(pb, gb):
    cx, cy = 0.5*(pb[0]+pb[2]), 0.5*(pb[1]+pb[3])
    return gb[0] <= cx <= gb[2] and gb[1] <= cy <= gb[3]


def match_greedy(preds, gts):
    """froc_eval.py per_image_match, verbatim. Score-descending; first UNMATCHED gt wins;
    every other prediction is a false positive."""
    ps = sorted(preds, key=lambda q: q['score'], reverse=True)
    matched = [False]*len(gts); out = []
    for q in ps:
        pb = [q['x_min'], q['y_min'], q['x_max'], q['y_max']]
        hit = -1
        for gi, gb in enumerate(gts):
            if not matched[gi] and center_in_box(pb, gb):
                hit = gi; break
        if hit >= 0:
            matched[hit] = True; out.append((q['score'], True))
        else:
            out.append((q['score'], False))
    return out, len(gts)


OPS = [0.125, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0]


def froc(gt, preds_by_img):
    """froc_eval.py froc_curve: pooled over images, cumulative, FP divided by len(gt) --
    which includes clean images, since load_gt gives them an empty box list."""
    all_tps, total = [], 0
    for img, gts in gt.items():
        m, n = match_greedy(preds_by_img.get(img, []), gts)
        all_tps.extend(m); total += n
    all_tps.sort(key=lambda x: -x[0])
    tp = fp = 0; sens = []; fpi = []
    for _, is_tp in all_tps:
        tp, fp = (tp+1, fp) if is_tp else (tp, fp+1)
        sens.append(tp/total if total else 0.0); fpi.append(fp/len(gt))
    return np.array(sens), np.array(fpi), total


def sens_at(fpi, sens, target):
    v = fpi <= target
    return float(sens[v].max()) if v.any() else 0.0


def ops_row(gt, preds):
    s, f, ngt = froc(gt, preds)
    pts = {o: sens_at(f, s, o) for o in OPS}
    return pts, float(np.mean(list(pts.values()))), ngt, len(gt), (s, f)


def preds_from(detdir, name, native=True):
    f = Path(detdir)/f'{name.replace(".mha", "")}.json'
    if not f.exists():
        return None
    d = json.load(open(f))
    b, sc = np.array(d['boxes']).reshape(-1, 4), np.array(d['scores'])
    return [dict(x_min=float(q[0]), y_min=float(q[1]), x_max=float(q[2]),
                 y_max=float(q[3]), score=float(s)) for q, s in zip(b, sc)]


def sync(sub):
    n = 0
    for f in (LOC/sub).iterdir():
        if not (REV/sub/f.name).exists():
            shutil.copy(f, REV/sub/f.name); n += 1
    return n


def restore(sub):
    n = 0
    for f in (REV/sub).iterdir():
        if not (LOC/sub/f.name).exists():
            shutil.copy(f, LOC/sub/f.name); n += 1
    return n


def stage(names, verbose=True):
    """Bulk copy before reading. Reading .mha one at a time over the mount stalls inside a
    C call where KeyboardInterrupt cannot reach it -- it has cost hours twice."""
    todo = [n for n in names if not (LOC/'node21'/n).exists()]
    if not todo:
        return LOC/'node21'
    t0 = time.time()
    for i, n in enumerate(todo):
        subprocess.run(['cp', str(MHA_SRC/n), str(LOC/'node21'/n)], check=True)
        if verbose and i % 250 == 0:
            print(f'  staged {i}/{len(todo)}  ({time.time()-t0:.0f}s)')
    if verbose:
        print(f'  staged {len(todo)} in {time.time()-t0:.0f}s')
    return LOC/'node21'


print(f'restored {restore("dets_native")} native + '
      f'{restore("dets_copypaste")} copy-paste detections from Drive')
print('machinery ready')

## 3 · Copy-paste twins — the controlled generator comparison · **item 3**

`copypaste.csv` has 648 rows matched to the grid by `radedit_twin`, and `det_edited` is
entirely NaN — generated and never scored. Same chests, same sites, same detector, same 512
canvas; **only the rendering differs**, so any gap is attributable to how the lesion was
rendered and no radiologist is needed to make the argument.

Deliberately uses the **512 path**, not native. The grid only exists at 512 because RadEdit
cannot run at any other resolution, so holding preprocessing constant across the two arms is
what makes the contrast valid. Both arms are equally out of distribution and it cancels.
These sensitivities must never be compared to the native real-data numbers in §6.

In [ ]:
cp = pd.read_csv(CPASTE/'copypaste.csv', keep_default_na=False, na_values=[''])
g  = pd.read_csv(GRID_CSV, keep_default_na=False, na_values=[''])
IDCOL = 'image_id' if 'image_id' in cp.columns else cp.columns[0]
BOX = next((c for c in [('mask_x0','mask_y0','mask_x1','mask_y1'),
                        ('x0','y0','x1','y1'), ('box_x0','box_y0','box_x1','box_y1')]
            if all(q in cp.columns for q in c)), None)
assert BOX, f'no recognisable box columns in {list(cp.columns)}'
print(f'{len(cp)} twins | id={IDCOL} | boxes={BOX}')

todo = [r for r in cp.itertuples()
        if not (LOC/'dets_copypaste'/f'{getattr(r, IDCOL)}.json').exists()]
print(f'{len(cp)-len(todo)} already scored, {len(todo)} to go')
t0 = time.time()
for i, r in enumerate(todo):
    iid = getattr(r, IDCOL)
    f = CPASTE/'images'/f'{iid}.png'
    assert f.exists(), f'{f} missing -- refusing to skip silently'
    a = np.asarray(Image.open(f).convert('L'), np.float32)/255.0
    if a.shape != (SIZE, SIZE):
        a = cv2.resize(a, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    b, sc = detect_512(a)
    json.dump({'boxes': b.tolist(), 'scores': sc.tolist()},
              open(LOC/'dets_copypaste'/f'{iid}.json', 'w'))
    if (i+1) % 100 == 0 or i == len(todo)-1:
        el = time.time()-t0
        print(f'  {i+1}/{len(todo)}  {el/60:.1f} min  +{sync("dets_copypaste")} to Drive')

def best_in(preds, box):
    return max((q['score'] for q in preds if center_in_box(
        [q['x_min'], q['y_min'], q['x_max'], q['y_max']], box)), default=0.0)

rows = []
for r in cp.itertuples():
    iid = getattr(r, IDCOL)
    pr = preds_from(LOC/'dets_copypaste', iid)
    if pr is None: continue
    rows.append(dict(image_id=iid, radedit_twin=r.radedit_twin,
                     det_cp=round(best_in(pr, [getattr(r, q) for q in BOX]), 4),
                     n_boxes_cp=len(pr)))
CP = pd.DataFrame(rows)
gi = g.set_index('image_id')
for c_, s_ in [('det_radedit', 'det_edited'), ('cnr_radedit', 'cnr'), ('chest', 'chest')]:
    CP[c_] = CP.radedit_twin.map(gi[s_])
PAIR = CP.dropna(subset=['det_radedit']).copy()
PAIR['hit_cp'] = PAIR.det_cp >= SCORE_MIN
PAIR['hit_re'] = PAIR.det_radedit >= SCORE_MIN
CP.to_csv(REV/'tables'/'copypaste_scored.csv', index=False)
print(f'\n{len(PAIR)} pairs over {PAIR.chest.nunique()} chests')
print(f'  copy-paste sensitivity @{SCORE_MIN}: {PAIR.hit_cp.mean():.4f} '
      f'({PAIR.hit_cp.sum()}/{len(PAIR)})')
print(f'  RadEdit    sensitivity @{SCORE_MIN}: {PAIR.hit_re.mean():.4f} '
      f'({PAIR.hit_re.sum()}/{len(PAIR)})')

In [ ]:
# paired test, with chest as the unit of independence
from scipy.stats import binomtest
b01 = int((~PAIR.hit_cp &  PAIR.hit_re).sum())
b10 = int(( PAIR.hit_cp & ~PAIR.hit_re).sum())
mcp = binomtest(b10, b01+b10, 0.5).pvalue if b01+b10 else np.nan

rng = np.random.default_rng(0)
chests = PAIR.chest.unique(); per = {c: PAIR[PAIR.chest == c] for c in chests}
boot = [(lambda s: s.hit_cp.mean()-s.hit_re.mean())(
            pd.concat([per[c] for c in rng.choice(chests, len(chests), replace=True)]))
        for _ in range(4000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
obs = PAIR.hit_cp.mean()-PAIR.hit_re.mean()

T_GEN = pd.DataFrame([dict(n_pairs=len(PAIR), n_chests=len(chests),
    sens_copypaste=round(PAIR.hit_cp.mean(), 4), sens_radedit=round(PAIR.hit_re.mean(), 4),
    difference=round(obs, 4), ci_lo=round(lo, 4), ci_hi=round(hi, 4),
    mcnemar_p=mcp, discordant_cp_only=b10, discordant_re_only=b01,
    operating_point=SCORE_MIN, preprocessing='512 grid-parity')])
T_GEN.to_csv(REV/'tables'/'table3-generator-comparison.csv', index=False)
PER_CHEST = PAIR.groupby('chest').agg(n=('hit_cp','size'), cp=('hit_cp','mean'),
                                      re=('hit_re','mean'))
PER_CHEST['delta'] = (PER_CHEST.cp-PER_CHEST.re).round(4)
PER_CHEST.to_csv(REV/'tables'/'table3b-generator-per-chest.csv')
print(T_GEN.to_string(index=False))
print(f'\ndiscordant: copy-paste only {b10}, RadEdit only {b01}   McNemar p={mcp:.6f}')
print(f'chest-clustered difference {obs:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  '
      f'excludes zero: {bool(lo > 0 or hi < 0)}')
print('\nITEM 3: a positive difference means the gap is GENERATOR RENDERING, since chests,')
print('sites, detector and preprocessing are all held constant. The synthetic sensitivity')
print('then characterises the generator-detector pair, not the detector, and the copy-paste')
print('rate is the ceiling insertion realism allows.')

## 4 · k-NN coverage · **item 9**

The draft's coverage figures — 0/29, 4/29, 23/29, 29/29 at four radii — are a radius sweep in
unnormalised 512-d space. In high dimensions pairwise distances concentrate, so a four-radius
sweep mostly reports where that band sits, and going 0/29 → 29/29 over a 4× range is the
signature of that concentration rather than of a coverage threshold.

Replaced with **k-NN recall**: for each real failure, is any of its k nearest synthetic
neighbours also a failure? Rank-based, so no radius and no distance scale. Against two nulls,
because a recall number alone is uninterpretable — label-shuffled gives what chance produces
at this base rate, random-neighbour isolates what the embedding geometry contributes.

In [ ]:
def load_emb(path):
    z = np.load(path, allow_pickle=False); k = list(z.keys())
    if len(k) == 1 and z[k[0]].ndim == 2:
        raise RuntimeError(f'{path.name} is a bare matrix with no ids')
    return {q: z[q].astype(np.float32).ravel() for q in k}

E_syn, E_real = load_emb(EMB_DIR/'synth_full.npz'), load_emb(EMB_DIR/'real_full.npz')
l2 = lambda M: M/np.clip(np.linalg.norm(M, axis=1, keepdims=True), 1e-8, None)

# real failure labels: an image fails if ANY of its nodules is missed (case level, and the
# embeddings are whole-image so there is one vector per image)
NAT = LOC/'dets_native'
real_ids, yr = [], []
for n in ANNOT:
    pr = preds_from(NAT, n)
    if pr is None or n.replace('.mha', '') not in E_real and n not in E_real:
        continue
    key = n if n in E_real else n.replace('.mha', '')
    miss = any(best_in(pr, gb) < SCORE_MIN for gb in GTB[n])
    real_ids.append(key); yr.append(miss)
yr = np.array(yr, bool)
if len(real_ids) < 50:
    print(f'only {len(real_ids)} real images have native detections -- run section 5 first,')
    print('then re-run this cell. Skipping k-NN for now.')
    KNN = REVK = None
else:
    syn = g[(g.arm == 'lesion') & (g.det_background == 0)]
    syn = syn[syn.image_id.isin(E_syn)]
    ys = (syn.det_edited < SCORE_MIN).values.astype(bool)
    R = l2(np.stack([E_real[i] for i in real_ids]))
    S = l2(np.stack([E_syn[i] for i in syn.image_id]))
    print(f'real {R.shape} failures {yr.sum()} ({yr.mean():.4f}) | '
          f'synth {S.shape} failures {ys.sum()} ({ys.mean():.4f})')
    ORD = np.argsort(-(R @ S.T), axis=1)
    rng = np.random.default_rng(0); KS = [1, 3, 5, 10, 20, 50]; ND = 200
    rec = lambda k, lab, o: lab[o[:, :k]].any(axis=1)[yr].mean()
    rows = []
    for k in KS:
        obs = rec(k, ys, ORD)
        sh = [rec(k, rng.permutation(ys), ORD) for _ in range(ND)]
        rd = [ys[rng.integers(0, len(ys), size=(len(real_ids), k))].any(axis=1)[yr].mean()
              for _ in range(ND)]
        rows.append(dict(k=k, recall=round(obs, 4), shuffled_mean=round(np.mean(sh), 4),
                         shuffled_p95=round(np.percentile(sh, 95), 4),
                         random_nbr_mean=round(np.mean(rd), 4),
                         lift=round(obs-np.mean(sh), 4),
                         exceeds_p95=bool(obs > np.percentile(sh, 95))))
        print(f'  k={k:3} recall {obs:.4f}  shuffled {np.mean(sh):.4f} '
              f'(p95 {np.percentile(sh,95):.4f})  rand-nbr {np.mean(rd):.4f}  '
              f'{"ABOVE" if rows[-1]["exceeds_p95"] else "within null"}')
    KNN = pd.DataFrame(rows)
    ORD_T = np.argsort(-(S @ R.T), axis=1)
    REVK = pd.DataFrame([dict(k=k, recall_synth_to_real=round(
        yr[ORD_T[:, :k]].any(axis=1)[ys].mean(), 4)) for k in KS])
    KNN.to_csv(REV/'tables'/'table4-knn-coverage.csv', index=False)
    REVK.to_csv(REV/'tables'/'table4b-knn-reverse.csv', index=False)
    print(f'\nITEM 9: recall above the shuffled p95 AND above random-neighbour means the')
    print('synthetic set covers real failures in a way the geometry supports. Inside the')
    print('null is itself a finding, and consistent with the CNR result: the grid varies')
    print('requested attributes while real difficulty varies continuously in conspicuity.')
    print(f'\nreverse direction (synth->real):')
    print(REVK.to_string(index=False))

## 5 · Score everything native — 4,881 images, ~50 min

All 1,133 annotated plus all 3,748 clean, with the preprocessing the checkpoint was trained
on. The clean images are what make a **true** false-positive rate possible: §5–§9 of notebook
03 divided false positives by annotated images only.

Resumable — one JSON per image, synced every 50. A dead session costs 50 images.

In [ ]:
ALL_IMGS = ANNOT + CLEAN
print(f'{len(ALL_IMGS)} images total')
stage(ALL_IMGS)

todo = [n for n in ALL_IMGS if not (LOC/'dets_native'/f'{n.replace(".mha","")}.json').exists()]
print(f'{len(ALL_IMGS)-len(todo)} already scored, {len(todo)} to go')
t0 = time.time()
for i, n in enumerate(todo):
    f = LOC/'node21'/n
    assert f.exists(), f'{f} missing -- staging incomplete, refusing to skip'
    a = load_native(f)
    b, sc = detect_native(a)
    json.dump({'boxes': b.tolist(), 'scores': sc.tolist(),
               'shape': [int(a.shape[0]), int(a.shape[1])]},
              open(LOC/'dets_native'/f'{n.replace(".mha","")}.json', 'w'))
    if (i+1) % 50 == 0 or i == len(todo)-1:
        el = time.time()-t0
        print(f'  {i+1}/{len(todo)}  {el/60:5.1f} min  '
              f'~{el/(i+1)*(len(todo)-i-1)/60:5.0f} min left  +{sync("dets_native")} to Drive')
n_have = len(list((LOC/'dets_native').glob('*.json')))
print(f'\n{n_have} native detection files, {sync("dets_native")} synced')
assert n_have >= len(ANNOT), 'annotated images are not all scored'

## 6 · Baseline 1 — FROC, `froc_eval.py` convention · **item 10**

`froc_eval.py` is the definition every historical number in this project came from and it
matches the NODE21 challenge convention: greedy one-to-one matching, and clean images in the
denominator via `gt[img] = []`.

Four scopes. **val and test are the honest generalisation numbers**; `all` pools the 907
training images the detector memorised, and `train` is reported only to show the size of that
effect. Wilson intervals on sensitivity at each operating point, since Table 1 had none.

In [ ]:
def build_gt_preds(img_names, include_clean=True):
    gt, pr = {}, {}
    for n in img_names:
        q = preds_from(LOC/'dets_native', n)
        if q is None: continue
        boxes = GTB.get(n, [])
        if not boxes and not include_clean: continue
        gt[n] = boxes; pr[n] = q
    return gt, pr


def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph, d = k/n, 1+z**2/n
    c = (ph+z**2/(2*n))/d
    h = z*np.sqrt(ph*(1-ph)/n + z**2/(4*n**2))/d
    return max(0.0, c-h), min(1.0, c+h)


rows, CURVES = [], {}
for scope in ['all', 'train', 'val', 'test']:
    names = (ALL_IMGS if scope == 'all'
             else [n for n in ALL_IMGS if SPLIT_OF.get(n) == scope])
    gt, pr = build_gt_preds(names)
    n_ann = sum(1 for v in gt.values() if v)
    if n_ann < 20:
        print(f'  {scope}: only {n_ann} annotated images, skipped'); continue
    pts, fmean, ngt, nimg, curve = ops_row(gt, pr)
    CURVES[scope] = curve
    r = dict(scope=scope, n_images=nimg, n_annotated=n_ann, n_nodules=ngt,
             froc_mean=round(fmean, 4))
    for o in OPS:
        lo, hi = wilson(int(round(pts[o]*ngt)), ngt)
        r[f'sens@{o}'] = round(pts[o], 4)
        r[f'ci@{o}'] = f'[{lo:.3f}, {hi:.3f}]'
    rows.append(r)
    print(f'  {scope:6} {nimg:5} img ({n_ann:4} annotated, {ngt:4} nodules)  '
          f'FROC {fmean:.4f}  sens@0.5 {pts[0.5]:.4f}')

T1 = pd.DataFrame(rows)
T1.to_csv(REV/'tables'/'table1-baseline1-froc.csv', index=False)
print()
print(T1[['scope','n_images','n_annotated','n_nodules','froc_mean']
        + [f'sens@{o}' for o in OPS]].to_string(index=False))

# the annotated-only denominator, for comparison with notebook 03 sections 5-9
gt_a, pr_a = build_gt_preds(ALL_IMGS, include_clean=False)
pts_a, f_a, _, n_a = ops_row(gt_a, pr_a)[:4]
print(f'\nannotated-only denominator ({n_a} images), as sections 5-9 of notebook 03 did:')
print(f'  FROC {f_a:.4f}  sens@0.5 {pts_a[0.5]:.4f}')
print(f'vs the froc_eval.py convention on all: '
      f'FROC {T1[T1.scope=="all"].froc_mean.iloc[0]:.4f}')
print('\nBoth definitional differences push sensitivity DOWN, so the froc_eval row being at')
print('or below the annotated-only row is expected. Report the froc_eval val and test rows.')
print('The 0.889 in the earlier draft is reachable by neither; treat it as unsourced.')

## 7 · Baseline 2 — abstention, corrected · **items 7 and 10**

Three faults in the original sweep, all fixed here: abstention was nodule-weighted so a
3-nodule image was abstained on three times; Wilson treated 1,474 nodules as independent; and
the operating point was inherited silently from a constant.

Reports the **case-level** rate as primary — the fraction of cases with at least one missed
nodule — because that is what a clinician acts on. Cluster bootstrap over images alongside
Wilson, and the comparison of the two is itself the finding.

In [ ]:
SCOPE_AB = 'val' if (SPL.split == 'val').any() else 'all'
ab_names = [n for n in ANNOT if SCOPE_AB == 'all' or SPLIT_OF.get(n) == SCOPE_AB]
recs = []
for n in ab_names:
    pr = preds_from(LOC/'dets_native', n)
    if pr is None: continue
    mx = max((q['score'] for q in pr), default=0.0)
    for j, gb in enumerate(GTB[n]):
        recs.append(dict(img_name=n, nodule=j, img_max=mx,
                         best=best_in(pr, gb)))
AB = pd.DataFrame(recs)
AB['missed'] = AB.best < SCORE_MIN
N_IMG_ALL = AB.img_name.nunique()
print(f'{SCOPE_AB}: {N_IMG_ALL} images, {len(AB)} nodules, '
      f'{len(AB)/N_IMG_ALL:.2f} nodules per image')

rng = np.random.default_rng(0); rows = []
for a in np.arange(0.0, 0.95, 0.05):
    ans = AB[AB.img_max >= a]
    if ans.img_name.nunique() == 0: continue
    grp = ans.groupby('img_name').missed.agg(['sum', 'count'])
    s, c = grp['sum'].to_numpy(float), grp['count'].to_numpy(float)
    n_i, n_n = len(grp), int(c.sum()); miss = int(s.sum())
    idx = rng.integers(0, n_i, size=(4000, n_i))
    num, den = s[idx].sum(1), c[idx].sum(1)
    bt = np.divide(num, den, out=np.full(4000, np.nan), where=den > 0)
    lo, hi = np.nanpercentile(bt, [2.5, 97.5])
    wlo, whi = wilson(miss, n_n)
    rows.append(dict(abstain_below=round(a, 2),
                     abstention_rate_cases=round(1-n_i/N_IMG_ALL, 4),
                     n_cases=n_i, n_nodules=n_n,
                     fnr_case=round((s > 0).sum()/n_i, 4),
                     fnr_nodule=round(miss/n_n, 4),
                     clustered_lo=round(lo, 4), clustered_hi=round(hi, 4),
                     wilson_lo=round(wlo, 4), wilson_hi=round(whi, 4),
                     ci_inflation=round((hi-lo)/(whi-wlo), 2) if whi > wlo else np.nan))
T2 = pd.DataFrame(rows)
T2.to_csv(REV/'tables'/'table2-baseline2-abstention.csv', index=False)
print(T2[['abstain_below','abstention_rate_cases','n_cases','fnr_case','fnr_nodule',
          'clustered_lo','clustered_hi','ci_inflation']].to_string(index=False))
z = T2.iloc[0]
print(f'\nat zero abstention: case FNR {z.fnr_case:.4f}, nodule FNR {z.fnr_nodule:.4f}')
print(f'median CI inflation over Wilson: {T2.ci_inflation.median():.2f}x')
print('\nITEM 10: inflation near 1.0 means Wilson is adequate here, because there are only')
print(f'{len(AB)/N_IMG_ALL:.2f} nodules per image. State that explicitly -- it is what')
print('justifies Wilson on the real side while chest-as-unit is load-bearing on the')
print('synthetic side, where cluster size is 60 rather than 1.3.')
print('ITEM 7: the threshold is chosen and evaluated on the same cases, and the answered')
print('subset is selected on img_max which correlates with best, so the FNR falls partly by')
print('construction. Closed by disclosure, not by fix.')

## 8 · Real-only predictor — the power check · **item 8**

The draft reports a real-only failure-predictor AUROC of 0.667 with CI 0.444–0.862 on 34
cases. That interval spans chance, so n=34 cannot support the comparison being asked of it.

This does not attempt to rescue it. It reports what n the current data supports and what
width an interval would have, so the claim can either be stated as underpowered with the
interval attached or dropped. **Do not confuse 0.667 with the FROC mean in §6** — coincidentally
similar numbers, entirely different quantities.

In [ ]:
n_fail = int(AB.groupby('img_name').missed.any().sum())
n_case = N_IMG_ALL
print(f'{SCOPE_AB}: {n_case} cases, {n_fail} with at least one missed nodule '
      f'({n_fail/n_case:.4f})')
lo34, hi34 = wilson(int(round(0.667*34)), 34)
print(f'\nthe draft: AUROC 0.667, CI 0.444-0.862, n=34 -- width {0.862-0.444:.3f}, '
      'spans chance (0.5)')
print(f'a Wilson interval on a rate of 0.667 at n=34 is [{lo34:.3f}, {hi34:.3f}] '
      f'-- width {hi34-lo34:.3f}')
rows = []
for n in [34, 50, 100, n_case]:
    lo, hi = wilson(int(round(0.667*n)), n)
    rows.append(dict(n=n, ci_lo=round(lo, 3), ci_hi=round(hi, 3),
                     width=round(hi-lo, 3), excludes_chance=bool(lo > 0.5)))
T5 = pd.DataFrame(rows)
T5.to_csv(REV/'tables'/'table5-real-only-power.csv', index=False)
print()
print(T5.to_string(index=False))
print(f'\nITEM 8: at n={n_case} available cases the interval '
      f"{'excludes' if T5.iloc[-1].excludes_chance else 'still includes'} chance.")
print('Either report 0.667 as underpowered with its interval attached, or drop the')
print('comparison. Do not report it bare.')

## 9 · Figures — the locked palette

Hex values sampled pixel by pixel from the approved figures and duplicated from
`src/figstyle.py`. **Do not substitute a matplotlib colormap.** Every figure is written at
300 DPI PNG plus vector PDF so it drops straight into LaTeX.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

INK, EDGE, GRIDC = '#2c3e50', '#1f2d4d', '#eeeeee'
BLUE, BLUE_MID, BLUE_DEEP, BLUE_LIT = '#6694de', '#4472c4', '#3c5182', '#5583d1'
BLUE_PALE, BLUE_MIST, BLUE_WASH = '#9ab2e5', '#c6d1eb', '#e6ebf5'
SLATE, SLATE_PALE, CORAL = '#7982a6', '#b4b8cb', '#fc7c7c'
CMAP = LinearSegmentedColormap.from_list('mdlf_blues',
    ['#f7f7f7', BLUE_MIST, BLUE_PALE, BLUE_LIT, BLUE_MID, BLUE_DEEP, '#33456e'])

plt.rcParams.update({'savefig.dpi': 300, 'figure.facecolor': 'white', 'font.size': 12,
    'axes.titleweight': 'bold', 'axes.titlecolor': INK, 'axes.labelweight': 'bold',
    'axes.labelcolor': INK, 'text.color': INK, 'xtick.color': INK, 'ytick.color': INK,
    'axes.edgecolor': INK, 'axes.spines.top': False, 'axes.spines.right': False,
    'grid.color': GRIDC, 'axes.axisbelow': True, 'legend.frameon': False})

FIGDIR = REV/'figures'


def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(FIGDIR/f'{name}.{ext}', dpi=300, bbox_inches='tight',
                    facecolor='white')
    print(f'  {name}.png + .pdf')


print('palette locked; figures ->', FIGDIR)

In [ ]:
# FIG 1 -- FROC by split, froc_eval convention
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
cols = {'all': BLUE_PALE, 'train': SLATE_PALE, 'val': BLUE_MID, 'test': BLUE_DEEP}
for scope, (s, f) in CURVES.items():
    m = f > 0
    ax[0].semilogx(f[m], s[m], color=cols.get(scope, BLUE), lw=2.4,
                   label=f'{scope} (FROC '
                         f'{T1[T1.scope==scope].froc_mean.iloc[0]:.3f})')
ax[0].set_xlabel('False positives per image'); ax[0].set_ylabel('Sensitivity')
ax[0].set_title('Baseline 1 — FROC by split'); ax[0].set_ylim(0, 1.02)
ax[0].grid(True, which='both', color=GRIDC); ax[0].legend(fontsize=9)

w = 0.19
xs = np.arange(len(OPS))
for k, scope in enumerate([s for s in ['val', 'test', 'all', 'train'] if s in CURVES]):
    r = T1[T1.scope == scope].iloc[0]
    ax[1].bar(xs + (k-1.5)*w, [r[f'sens@{o}'] for o in OPS], w,
              color=cols.get(scope, BLUE), edgecolor=EDGE, lw=.6, label=scope)
ax[1].set_xticks(xs); ax[1].set_xticklabels([str(o) for o in OPS])
ax[1].set_xlabel('False positives per image'); ax[1].set_ylabel('Sensitivity')
ax[1].set_title('Sensitivity at the seven operating points')
ax[1].set_ylim(0, 1.02); ax[1].grid(True, axis='y', color=GRIDC); ax[1].legend(fontsize=9)
plt.suptitle('Baseline 1 on NODE21 — froc_eval.py convention',
             fontsize=16, fontweight='bold', color=INK)
plt.tight_layout(rect=[0, 0, 1, 0.93]); save(fig, 'fig1-baseline1-froc'); plt.show()

In [ ]:
# FIG 2 -- abstention, case vs nodule, clustered vs Wilson
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
x = T2.abstention_rate_cases
ax[0].fill_between(x, T2.wilson_lo, T2.wilson_hi, color=SLATE_PALE, alpha=.45,
                   label='Wilson')
ax[0].fill_between(x, T2.clustered_lo, T2.clustered_hi, color=SLATE, alpha=.22,
                   label='image-clustered bootstrap')
ax[0].plot(x, T2.fnr_nodule, color=SLATE, lw=2.4, marker='o', ms=4, label='per nodule')
ax[0].plot(x, T2.fnr_case, color=BLUE_DEEP, lw=2.2, ls='--', marker='s', ms=3.5,
           label='per case (≥1 missed)')
ax[0].set_xlabel('Abstention rate (cases declined)')
ax[0].set_ylabel(f'FNR on answered cases, OP={SCORE_MIN}')
ax[0].set_title('Baseline 2 — confidence-only abstention')
ax[0].grid(True, color=GRIDC); ax[0].legend(fontsize=8)

ax[1].plot(x, T2.ci_inflation, color=BLUE_MID, lw=2.4, marker='o', ms=4)
ax[1].axhline(1.0, color=CORAL, lw=1.6, ls=':')
ax[1].annotate('no inflation — Wilson is adequate', (x.iloc[len(x)//2], 1.0),
               textcoords='offset points', xytext=(0, 8), fontsize=9, color=CORAL)
ax[1].set_xlabel('Abstention rate (cases declined)')
ax[1].set_ylabel('Clustered CI width ÷ Wilson width')
ax[1].set_title(f'Clustering correction — {len(AB)/N_IMG_ALL:.2f} nodules per image')
ax[1].grid(True, color=GRIDC)
plt.suptitle(f'Abstention on NODE21 ({SCOPE_AB})', fontsize=16, fontweight='bold', color=INK)
plt.tight_layout(rect=[0, 0, 1, 0.93]); save(fig, 'fig2-abstention'); plt.show()

In [ ]:
# FIG 3 -- generator comparison, item 3
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
ax[0].bar([0, 1], [PAIR.hit_cp.mean(), PAIR.hit_re.mean()], 0.55,
          color=[BLUE_MID, SLATE], edgecolor=EDGE, lw=.8)
for i, v in enumerate([PAIR.hit_cp.mean(), PAIR.hit_re.mean()]):
    ax[0].text(i, v+.015, f'{v:.3f}', ha='center', fontweight='bold', color=INK)
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(['copy-paste\n(real patch)',
                                                 'RadEdit\n(diffusion)'])
ax[0].set_ylabel(f'Sensitivity @{SCORE_MIN}')
ax[0].set_title(f'Rendering is the only difference  (n={len(PAIR)} twins)')
ax[0].set_ylim(0, 1.02); ax[0].grid(True, axis='y', color=GRIDC)

pc = PER_CHEST.sort_values('delta')
yy = np.arange(len(pc))
ax[1].barh(yy, pc.delta, color=[BLUE_MID if d > 0 else CORAL for d in pc.delta],
           edgecolor=EDGE, lw=.6)
ax[1].axvline(0, color=INK, lw=1)
ax[1].axvline(T_GEN.ci_lo.iloc[0], color=SLATE, ls=':', lw=1.4)
ax[1].axvline(T_GEN.ci_hi.iloc[0], color=SLATE, ls=':', lw=1.4)
ax[1].set_yticks(yy); ax[1].set_yticklabels(pc.index, fontsize=9)
ax[1].set_xlabel('copy-paste − RadEdit sensitivity')
ax[1].set_title(f'Per chest, with the clustered 95% CI  '
                f'({T_GEN.difference.iloc[0]:+.3f})')
ax[1].grid(True, axis='x', color=GRIDC)
plt.suptitle('Controlled generator comparison — 512 px grid parity',
             fontsize=16, fontweight='bold', color=INK)
plt.tight_layout(rect=[0, 0, 1, 0.93]); save(fig, 'fig3-generator-comparison'); plt.show()

In [ ]:
# FIG 4 -- k-NN coverage, item 9
if KNN is not None:
    fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
    ax[0].plot(KNN.k, KNN.recall, color=BLUE_DEEP, lw=2.6, marker='o', ms=5,
               label='observed')
    ax[0].plot(KNN.k, KNN.shuffled_mean, color=SLATE, lw=2.0, ls='--', marker='s', ms=4,
               label='label-shuffled null')
    ax[0].fill_between(KNN.k, KNN.shuffled_mean, KNN.shuffled_p95, color=SLATE_PALE,
                       alpha=.5, label='shuffled mean → p95')
    ax[0].plot(KNN.k, KNN.random_nbr_mean, color=BLUE_PALE, lw=2.0, ls=':', marker='^',
               ms=4, label='random-neighbour null')
    ax[0].set_xscale('log'); ax[0].set_xticks(KNN.k)
    ax[0].set_xticklabels([str(int(v)) for v in KNN.k])
    ax[0].set_xlabel('k nearest synthetic neighbours')
    ax[0].set_ylabel('Fraction of real failures with a synthetic failure in top-k')
    ax[0].set_title('k-NN recall, real → synthetic')
    ax[0].set_ylim(0, 1.02); ax[0].grid(True, color=GRIDC); ax[0].legend(fontsize=8)

    ax[1].bar(np.arange(len(KNN)), KNN.lift, 0.6,
              color=[BLUE_MID if e else SLATE_PALE for e in KNN.exceeds_p95],
              edgecolor=EDGE, lw=.6)
    ax[1].axhline(0, color=INK, lw=1)
    ax[1].set_xticks(np.arange(len(KNN)))
    ax[1].set_xticklabels([str(int(v)) for v in KNN.k])
    ax[1].set_xlabel('k'); ax[1].set_ylabel('Recall − shuffled mean')
    ax[1].set_title('Lift over chance (filled = above shuffled p95)')
    ax[1].grid(True, axis='y', color=GRIDC)
    plt.suptitle('Coverage of real failures by the synthetic grid',
                 fontsize=16, fontweight='bold', color=INK)
    plt.tight_layout(rect=[0, 0, 1, 0.93]); save(fig, 'fig4-knn-coverage'); plt.show()
else:
    print('k-NN skipped in section 4 -- rerun that cell now that section 5 has finished.')

## 10 · Manifest, and every checklist number in one place

md5 of every output plus a one-screen summary to paste into the response letter. This is the
cell to screenshot.

In [ ]:
man = []
for sub in ['tables', 'figures']:
    for f in sorted((REV/sub).iterdir()):
        if f.is_file():
            man.append(dict(file=f'{sub}/{f.name}', bytes=f.stat().st_size, md5=md5(f)))
for f in [REV/'environment.json', REV/'requirements-lock.txt']:
    if f.exists():
        man.append(dict(file=f.name, bytes=f.stat().st_size, md5=md5(f)))
MAN = pd.DataFrame(man)
MAN.to_csv(REV/'MANIFEST.csv', index=False)
print(MAN.to_string(index=False))

print('\n' + '='*78)
print('REVISION SUMMARY — every number, ready to cite')
print('='*78)
for scope in ['val', 'test', 'all']:
    if scope in set(T1.scope):
        r = T1[T1.scope == scope].iloc[0]
        print(f'  Baseline 1 FROC ({scope:5}) {r.froc_mean:.4f}   '
              f'sens@0.5 {r["sens@0.5"]:.4f} {r["ci@0.5"]}   '
              f'n={int(r.n_images)} img / {int(r.n_nodules)} nodules')
z = T2.iloc[0]
print(f'  Baseline 2, zero abstention: case FNR {z.fnr_case:.4f}, '
      f'nodule FNR {z.fnr_nodule:.4f}')
print(f'  CI inflation over Wilson (median) {T2.ci_inflation.median():.2f}x  '
      f'-> Wilson {"adequate" if T2.ci_inflation.median() < 1.15 else "insufficient"}')
gr = T_GEN.iloc[0]
print(f'  ITEM 3  copy-paste {gr.sens_copypaste:.4f} vs RadEdit {gr.sens_radedit:.4f}  '
      f'diff {gr.difference:+.4f} [{gr.ci_lo:+.4f}, {gr.ci_hi:+.4f}] over '
      f'{int(gr.n_chests)} chests')
if KNN is not None:
    b = KNN.sort_values('lift', ascending=False).iloc[0]
    print(f'  ITEM 9  best k={int(b.k)}: recall {b.recall:.4f} vs shuffled '
          f'{b.shuffled_mean:.4f} (lift {b.lift:+.4f}), above p95 at any k: '
          f'{bool(KNN.exceeds_p95.any())}')
print(f'  ITEM 8  n={N_IMG_ALL} cases available; the draft\'s n=34 interval spans chance')
print(f'  ITEM 10 Wilson intervals on every operating point: table1')
print(f'  ITEM 7  abstention evaluated on fitted cases -- closed by disclosure')
print('\n  STILL NEEDS A PERSON: where the 0.889 sensitivity came from. No recovered')
print('  script produces it and both definitional differences push sensitivity down.')
print(f'\n  reproducibility: checkpoint md5 {ENV["checkpoint_md5"][:12]}, '
      f'split md5 {ENV["split_md5"][:12]}, {len(MAN)} artefacts hashed')
print(f'\n  everything at {REV}')

## 11 · Baseline 3 — unattended, last

Everything above is finished and saved before this starts, so a session death here costs only
the training run.

Recipe recovered from `train_baseline.py` — COCO init, batch 2, seed 42, SGD 0.005/0.9/5e-4,
StepLR 3/0.1, 5 epochs — and the split is the recovered one, so **Baseline 1's checkpoint is
a valid comparator** and only this one arm needs training.

6–10 GPU-hours against a 12-hour cap. Checkpoints every epoch to Drive; re-running resumes
from the last one. The full implementation is in `05_baseline3_augmentation.ipynb`; run it
after this notebook finishes rather than duplicating it here, since it needs the upstream
`training_utils` unzipped and a clean runtime for the long haul.

In [ ]:
print('Sections 1-10 are complete and saved. For Baseline 3:')
print()
print('  1. Open notebooks/05_baseline3_augmentation.ipynb')
print('  2. It reuses the same recovered recipe and split as this notebook')
print(f'  3. It reads the native Baseline 1 comparator from {REV}/tables/')
print('     table1-baseline1-froc.csv -- the val row')
print('  4. Fresh runtime, so the 12-hour cap starts from zero')
print()
print('Why not inline here: it needs the upstream node21_detection_baseline training_utils')
print('unzipped and imported, and a 6-10 hour run should not share a session with work that')
print('is already finished. Nothing in the revisions depends on its result.')
print()
for k, v in dict(epochs=5, batch=2, lr=0.005, momentum=0.9, weight_decay=0.0005,
                 lr_step=3, lr_gamma=0.1, seed=42, weights='DEFAULT (COCO)').items():
    print(f'  {k:14} {v}')
print(f'  split          {SPLIT_CSV.name}')